# AI Agent Demos — What These Systems Can Do For Your Business

**5 live examples. Each takes your input and produces a typed, structured result in under 10 seconds.**

Run all cells top to bottom. Replace the sample inputs with your own data at any time.

---

> These agents don't just return text — they return *decisions*: structured, consistent, and ready to plug into your workflows, your CRM, your ticketing system, or your reporting layer.

In [ ]:
%pip install -q langchain-openai langchain-core pydantic python-dotenv

import os

# ── Paste your OpenAI API key below ───────────────────────────────────────────
os.environ['OPENAI_API_KEY'] = 'sk-...'
# ──────────────────────────────────────────────────────────────────────────────

print('Setup complete. Ready to run demos.')

---

## Demo 1 — Email Triage

Imagine your team receives 200 emails a day. Some are on fire — a production outage, an angry enterprise client, a blocked payment. Others are routine. A handful are noise.

Right now, a human reads all of them to decide what matters. This agent does that work instantly — categorising each email, flagging urgency, and telling the right person what to do next.

**What it produces for each email:**
- Category (billing / technical / general / spam)
- Urgency level (high / medium / low)
- One-line summary
- Recommended next action

In [ ]:
# ── Demo 1: Email Triage ──────────────────────────────────────────────────────

from typing import Literal
from pydantic import BaseModel
from langchain_openai import ChatOpenAI


class EmailTriage(BaseModel):
    urgency: Literal["high", "medium", "low"]
    category: Literal["billing", "technical", "general", "spam"]
    summary: str
    recommended_action: str


def triage_email(email_text: str) -> EmailTriage:
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    chain = llm.with_structured_output(EmailTriage)
    return chain.invoke(
        f"Classify this email and recommend the next action:\n\n{email_text}"
    )


# ── Sample emails ─────────────────────────────────────────────────────────────
# Replace any of these with real emails from your inbox.

EMAILS = [
    {
        "label": "IT Outage",
        "text": (
            "From: marcos.reyes@acmecorp.com\n"
            "Subject: URGENT — API completely down, blocking 40 users\n\n"
            "Hi, our entire team has been locked out of the dashboard since 08:15 this morning. "
            "Every API call is returning a 503. We have a client presentation in 90 minutes "
            "and we cannot pull any data. This is costing us real money right now. "
            "Please escalate immediately — who do I call?"
        ),
    },
    {
        "label": "Invoice Request",
        "text": (
            "From: ap@northstarlegal.com\n"
            "Subject: Invoice #INV-2048 — payment processing\n\n"
            "Hello, I am the accounts payable coordinator at Northstar Legal. "
            "Could you resend the invoice for INV-2048 dated May 1st? "
            "Our records show it was sent to the wrong email address. "
            "Payment terms are net-30 so this is not urgent, just want to get it in the queue. Thanks."
        ),
    },
    {
        "label": "Sales Inquiry",
        "text": (
            "From: priya.mehta@greystone.io\n"
            "Subject: Interested in enterprise pricing\n\n"
            "Hi there, I came across your platform and I think it could be a great fit "
            "for our operations team of about 120 people. We are currently spending "
            "around $8k/month on a legacy tool and looking to switch before Q4. "
            "Can someone walk me through the enterprise tier and any volume discounts? "
            "Happy to do a call this week."
        ),
    },
]

# ── Run and print ─────────────────────────────────────────────────────────────

URGENCY_ICON = {"high": "[HIGH]", "medium": "[MED]", "low": "[LOW]"}

print("=" * 64)
print("  EMAIL TRIAGE RESULTS")
print("=" * 64)

for item in EMAILS:
    result = triage_email(item["text"])
    urgency_tag = URGENCY_ICON.get(result.urgency, result.urgency.upper())
    print(f"\n  Email: {item['label']}")
    print(f"  {'─' * 40}")
    print(f"  Category   : {result.category.upper()}")
    print(f"  Urgency    : {urgency_tag} {result.urgency.upper()}")
    print(f"  Summary    : {result.summary}")
    print(f"  Next Action: {result.recommended_action}")

print("\n" + "=" * 64)

---

## Demo 2 — Lead Qualification

Your sales team shouldn't spend equal time on every inbound lead. But deciding which ones deserve a call, which ones go into a nurture sequence, and which ones to drop — that takes judgment that most teams apply inconsistently.

This agent applies your Ideal Customer Profile criteria to every lead and returns a structured score with the reasoning made explicit. Every rep sees the same logic. Every lead gets a fair assessment.

**What it produces for each lead:**
- ICP fit score (1–10)
- Tier classification (hot / warm / cold)
- Which criteria were met and which were missed
- Recommended next action and rationale

In [ ]:
# ── Demo 2: Lead Qualification ────────────────────────────────────────────────

from typing import List, Literal
from pydantic import BaseModel, Field
from langchain_core.messages import SystemMessage
from langchain_openai import ChatOpenAI


class LeadScore(BaseModel):
    company: str
    score: int = Field(ge=1, le=10, description="ICP fit score from 1 (no fit) to 10 (perfect fit)")
    tier: Literal["hot", "warm", "cold"]
    criteria_met: List[str]
    criteria_missed: List[str]
    recommended_action: str
    reasoning: str


ICP_SYSTEM = SystemMessage("""
You are a sales qualification assistant. Score inbound leads against this ICP rubric.

IDEAL CUSTOMER PROFILE (ICP):
  Industry:      SaaS, FinTech, or E-commerce
  Company size:  50-500 employees
  Pain point:    manual workflows, data silos, or compliance burden
  Buyer role:    VP Operations, CFO, or CTO
  Budget signal: existing software spend > $5k/month

SCORING:
  8-10 -> hot   (3+ criteria met, strong pain + budget signal)
  5-7  -> warm  (2 criteria met, or strong pain but budget unclear)
  1-4  -> cold  (fewer than 2 criteria met)

Rules:
- Populate criteria_met and criteria_missed by naming the exact ICP criteria above.
- reasoning must explain the score in one or two sentences citing the criteria.
- Never invent data not present in the lead description.
""")


def qualify_lead(description: str) -> LeadScore:
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    chain = ICP_SYSTEM | llm.with_structured_output(LeadScore)
    return chain.invoke(description)


# ── Sample leads ──────────────────────────────────────────────────────────────
# Paste your own lead descriptions here — LinkedIn profiles, form submissions,
# CRM notes, call summaries. Whatever you have.

LEADS = [
    {
        "label": "Strong Fit",
        "description": (
            "Company: Finvera Technologies. 180 employees. B2B SaaS for insurance brokers. "
            "Contact: Elena Wurst, CFO. "
            "She filled out our demo form and mentioned that their finance team is drowning "
            "in manual reconciliation — three FTEs doing spreadsheet work every month-end. "
            "They currently pay $12k/month across five different tools they're trying to consolidate. "
            "Need to move before their fiscal year starts in October."
        ),
    },
    {
        "label": "Weak Fit",
        "description": (
            "Company: Pebble & Stone Photography. 8 employees. Event and portrait studio. "
            "Contact: Jordan Pebble, owner. "
            "Found us through a Google ad. Interested in any tools that can help with scheduling. "
            "Currently using free tier of Calendly. No mention of software budget or specific pain points "
            "beyond 'it would be nice to have things more organised'."
        ),
    },
]

# ── Run and print ─────────────────────────────────────────────────────────────

TIER_LABEL = {"hot": "HOT", "warm": "WARM", "cold": "COLD"}

print("=" * 64)
print("  LEAD QUALIFICATION RESULTS")
print("=" * 64)

for lead in LEADS:
    result = qualify_lead(lead["description"])
    tier_label = TIER_LABEL.get(result.tier, result.tier.upper())
    print(f"\n  Lead     : {lead['label']} — {result.company}")
    print(f"  {'─' * 40}")
    print(f"  Score    : {result.score}/10  |  Tier: {tier_label}")
    print(f"  Criteria Met   : {', '.join(result.criteria_met) or 'None'}")
    print(f"  Criteria Missed: {', '.join(result.criteria_missed) or 'None'}")
    print(f"  Reasoning      : {result.reasoning}")
    print(f"  Next Action    : {result.recommended_action}")

print("\n" + "=" * 64)

---

## Demo 3 — Contract Risk Review

Every contract your business signs carries risk. Most of that risk lives in the clauses that your team skims, or in the protections that are simply missing.

Sending every agreement to outside counsel costs hundreds of dollars and days of waiting. This agent reads contracts the way a senior commercial lawyer would — identifying risky clauses, flagging missing protections, and giving you a prioritised negotiation agenda — in seconds.

**What it produces for each contract:**
- Overall risk rating (high / medium / low)
- Executive summary for leadership
- Risk findings with clause citations and recommended redlines
- Missing standard protections with suggested language
- Prioritised negotiation agenda (must-have / should-have / nice-to-have)

In [ ]:
# ── Demo 3: Contract Risk Review ──────────────────────────────────────────────

from typing import List, Literal, Optional
from pydantic import BaseModel, Field
from langchain_core.messages import SystemMessage
from langchain_openai import ChatOpenAI


class RiskFinding(BaseModel):
    severity: Literal["critical", "high", "medium", "low"]
    category: Literal[
        "liability", "payment", "ip", "termination",
        "confidentiality", "governance", "compliance", "other",
    ]
    clause_reference: str
    issue: str
    implication: str
    recommended_redline: str


class MissingProtection(BaseModel):
    protection: str
    why_needed: str
    suggested_clause: str


class NegotiationPoint(BaseModel):
    priority: Literal["must_have", "should_have", "nice_to_have"]
    topic: str
    current_position: str
    target_position: str


class ContractReview(BaseModel):
    contract_type: str
    counterparty: Optional[str] = None
    governing_law: Optional[str] = None
    overall_risk: Literal["high", "medium", "low"]
    executive_summary: str
    risk_findings: List[RiskFinding]
    missing_protections: List[MissingProtection]
    negotiation_points: List[NegotiationPoint]


REVIEWER_SYSTEM = SystemMessage("""
You are a senior commercial lawyer reviewing a contract on behalf of a client.

Your analysis must cover three areas:

1. RISK FINDINGS -- every clause that creates risk for the client
   - You MUST include a clause_reference (section/clause number) for every finding
   - Quote or closely paraphrase the language that creates the risk
   - recommended_redline must be concrete proposed language, not vague advice

2. MISSING PROTECTIONS -- standard clauses absent from this contract
   - Only flag protections that are genuinely standard for this contract type
   - Provide draft suggested_clause language the client can propose

3. NEGOTIATION POINTS -- prioritised list of what to push for
   - must_have: deal-breakers; walk away if not addressed
   - should_have: important but not fatal
   - nice_to_have: improvements worth raising if the counterparty is cooperative
""")


def review_contract(contract_text: str) -> ContractReview:
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    chain = REVIEWER_SYSTEM | llm.with_structured_output(ContractReview)
    return chain.invoke(f"CONTRACT TO REVIEW:\n\n{contract_text}")


# ── Sample contract excerpt ───────────────────────────────────────────────────
# Paste your own contract text here — MSA, SaaS agreement, services contract.

SAMPLE_CONTRACT = """
MASTER SERVICES AGREEMENT
Between: Orion Data Solutions Inc. ("Vendor") and Customer
Governing Law: State of Delaware

Section 3. PAYMENT TERMS
Customer agrees to pay all invoices within 15 days of receipt. Vendor reserves the right
to charge interest at 2.5% per month on any overdue balances. Vendor may suspend
services without notice if any invoice remains unpaid for more than 10 days.

Section 5. INTELLECTUAL PROPERTY
All work product, deliverables, and custom configurations developed by Vendor under this
Agreement shall remain the sole and exclusive property of Vendor. Customer receives a
non-exclusive, non-transferable license to use the deliverables solely during the term
of this Agreement.

Section 7. INDEMNIFICATION
Customer shall indemnify, defend, and hold harmless Vendor and its officers, employees,
and agents from and against any and all claims, damages, losses, and expenses arising
out of or related to Customer's use of the Services, including reasonable attorneys' fees.

Section 8. LIMITATION OF LIABILITY
IN NO EVENT SHALL VENDOR BE LIABLE FOR ANY INDIRECT, INCIDENTAL, SPECIAL, OR
CONSEQUENTIAL DAMAGES. VENDOR'S TOTAL LIABILITY FOR ANY CLAIM ARISING UNDER
THIS AGREEMENT SHALL NOT EXCEED THE AMOUNTS PAID BY CUSTOMER IN THE
THREE (3) MONTHS PRECEDING THE CLAIM.

Section 10. TERMINATION
Vendor may terminate this Agreement for any reason upon 7 days written notice.
Customer may terminate only for Vendor's uncured material breach after providing
60 days written notice and cure period.

Section 12. DATA HANDLING
Vendor will process Customer data as necessary to provide the Services. Customer
grants Vendor a perpetual license to use anonymised Customer data for product
improvement and benchmarking purposes.
"""

# ── Run and print ─────────────────────────────────────────────────────────────

result = review_contract(SAMPLE_CONTRACT)

RISK_LABEL = {"high": "HIGH RISK", "medium": "MEDIUM RISK", "low": "LOW RISK"}
SEV_LABEL  = {"critical": "[CRITICAL]", "high": "[HIGH]", "medium": "[MEDIUM]", "low": "[LOW]"}
PRI_LABEL  = {"must_have": "[MUST HAVE]", "should_have": "[SHOULD HAVE]", "nice_to_have": "[NICE TO HAVE]"}

print("=" * 64)
print("  CONTRACT RISK REVIEW")
print("=" * 64)
print(f"\n  Contract Type : {result.contract_type}")
print(f"  Counterparty  : {result.counterparty or 'Not identified'}")
print(f"  Governing Law : {result.governing_law or 'Not stated'}")
print(f"  Overall Risk  : {RISK_LABEL.get(result.overall_risk, result.overall_risk.upper())}")
print(f"\n  EXECUTIVE SUMMARY\n  {'─' * 40}")
print(f"  {result.executive_summary}")

print(f"\n  RISK FINDINGS ({len(result.risk_findings)} found)\n  {'─' * 40}")
for i, f in enumerate(result.risk_findings, 1):
    sev = SEV_LABEL.get(f.severity, f.severity.upper())
    print(f"\n  [{i}] {sev} -- {f.clause_reference} ({f.category})")
    print(f"       Issue    : {f.issue}")
    print(f"       Impact   : {f.implication}")
    print(f"       Redline  : {f.recommended_redline}")

print(f"\n  MISSING PROTECTIONS ({len(result.missing_protections)} found)\n  {'─' * 40}")
for m in result.missing_protections:
    print(f"\n  * {m.protection}")
    print(f"    Why needed : {m.why_needed}")

print(f"\n  NEGOTIATION AGENDA\n  {'─' * 40}")
for n in result.negotiation_points:
    p_label = PRI_LABEL.get(n.priority, n.priority.upper())
    print(f"\n  {p_label} {n.topic}")
    print(f"     Now    : {n.current_position}")
    print(f"     Target : {n.target_position}")

print("\n" + "=" * 64)

---

## Demo 4 — Executive Assistant

Senior executives and their assistants spend hours each week doing the same three things with email: deciding what to reply, figuring out who owns what, and tracking loose threads so nothing falls through the cracks.

This agent reads an email thread the same way a world-class assistant would. It drafts the reply, pulls out every action item with the named owner and deadline, and builds a follow-up tracker for anything still in motion.

**What it produces for each thread:**
- Polished draft reply, ready to send
- Suggested subject line
- Action items with owners, deadlines, and priority
- Follow-up tracker for pending items

In [ ]:
# ── Demo 4: Executive Assistant ───────────────────────────────────────────────

from typing import List, Literal, Optional
from pydantic import BaseModel, Field
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_openai import ChatOpenAI


class ActionItem(BaseModel):
    description: str
    owner: Optional[str] = None
    due_date: Optional[str] = None
    priority: Literal["high", "medium", "low"]


class FollowUpEntry(BaseModel):
    topic: str
    waiting_on: Optional[str] = None
    check_in_by: Optional[str] = None
    notes: str


class ExecOutput(BaseModel):
    input_type: Literal["email_thread", "meeting_transcript"]
    draft_reply: str
    subject_line: Optional[str] = None
    action_items: List[ActionItem]
    follow_up_tracker: List[FollowUpEntry]
    meeting_summary: Optional[str] = None


ASSISTANT_SYSTEM = SystemMessage("""
You are a world-class executive assistant.

From the input (either an email thread or a meeting transcript) produce ALL of the
following in one pass:

1. draft_reply      -- A polished, ready-to-send reply or acknowledgement.
2. action_items     -- Every discrete task mentioned or implied. Name the owner and
                       deadline where the text makes them identifiable. Priority:
                       "high" = blocking or time-sensitive, "medium" = this week,
                       "low" = nice-to-have.
3. follow_up_tracker -- Items that need monitoring but are not direct tasks yet.
4. meeting_summary  -- Populate only if the input is a meeting transcript.
5. subject_line     -- Populate only if the input is an email thread.

Set input_type = "email_thread" or "meeting_transcript" based on the content.
Every output must be populated -- no empty lists.
""")


def run_exec_assistant(input_text: str) -> ExecOutput:
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    chain = ASSISTANT_SYSTEM | llm.with_structured_output(ExecOutput)
    return chain.invoke(HumanMessage(content="Input to process:\n\n" + input_text))


# ── Sample email thread ───────────────────────────────────────────────────────
# Replace with a real thread from your inbox -- any topic works.

EMAIL_THREAD = """
--- Email 1 ---
From: claire.nakamura@acme.com
To: ryan.okafor@acme.com, priya.suresh@acme.com
Date: Monday, 16 Jun -- 9:04 AM
Subject: Product launch -- Horizon v2 -- timeline check

Team, quick check-in on where we stand for the Horizon v2 launch scheduled for July 14th.
We need to lock the press release copy by this Friday (Jun 20). Priya, can you own that
draft and have a first pass in my inbox by Thursday EOD?

Ryan -- the pricing page update was supposed to be live last week. Still seeing the old
numbers. Can you confirm with the web team when this will actually go live? We need it
done before the launch blog goes out on July 10.

Also flagging: we still don't have sign-off from Legal on the data processing addendum
for EU customers. I pinged Marcus on Friday but haven't heard back. If we don't have
this by June 27 we may have to delay the EU rollout.

--- Email 2 ---
From: ryan.okafor@acme.com
To: claire.nakamura@acme.com, priya.suresh@acme.com
Date: Monday, 16 Jun -- 10:22 AM

Claire, on the pricing page -- web team says it's in the deploy queue for Wednesday.
I'll confirm once it's live. If it slips again I'll escalate to Jess.

Also worth flagging: the in-app announcement banner still needs copy. I don't think
anyone has picked that up. Probably 2 hours of work but needs to happen before July 8.

--- Email 3 ---
From: priya.suresh@acme.com
To: claire.nakamura@acme.com, ryan.okafor@acme.com
Date: Monday, 16 Jun -- 11:45 AM

On the press release -- understood, I'll have a draft to Claire by Thursday 5pm.

For the in-app banner, happy to take that on too if nobody else is. Will aim for
June 30 to leave buffer. Can someone confirm what the character limit is?

One more thing: the demo video we planned for the launch blog -- has anyone checked
if the production team is still on track for a June 25 delivery? I know they had
capacity issues last month.
"""

# ── Run and print ─────────────────────────────────────────────────────────────

result = run_exec_assistant(EMAIL_THREAD)

PRI_LABEL = {"high": "[HIGH]", "medium": "[MED]", "low": "[LOW]"}

print("=" * 64)
print("  EXECUTIVE ASSISTANT OUTPUT")
print("=" * 64)

if result.subject_line:
    print(f"\n  Subject Line: {result.subject_line}")

print(f"\n  DRAFT REPLY\n  {'─' * 40}")
for line in result.draft_reply.split("\n"):
    print(f"  {line}")

print(f"\n  ACTION ITEMS ({len(result.action_items)} found)\n  {'─' * 40}")
for i, item in enumerate(result.action_items, 1):
    p_label = PRI_LABEL.get(item.priority, item.priority.upper())
    owner  = item.owner    or "Unassigned"
    due    = item.due_date or "No deadline stated"
    print(f"\n  [{i}] {p_label} {item.description}")
    print(f"       Owner : {owner}")
    print(f"       Due   : {due}")

print(f"\n  FOLLOW-UP TRACKER\n  {'─' * 40}")
for f in result.follow_up_tracker:
    waiting  = f.waiting_on  or "Unknown"
    check_in = f.check_in_by or "Not set"
    print(f"\n  * {f.topic}")
    print(f"    Waiting on : {waiting}")
    print(f"    Check in by: {check_in}")
    print(f"    Notes      : {f.notes}")

print("\n" + "=" * 64)

---

## What happens next?

Every demo above runs against **your real data** — just swap the sample text for actual emails, leads, or contracts from your business.

These agents aren't prototypes. They're the same architecture used in production systems handling thousands of decisions per day. What you've seen here can be:

- **Connected to your existing tools** — Salesforce, HubSpot, Zendesk, Slack, your internal CRM
- **Automated end-to-end** — inputs arrive, outputs are written back to your system, no human in the loop
- **Customised to your business logic** — your criteria, your categories, your tone
- **Expanded** — these four patterns are a fraction of what's available

### Full example library

All examples, source code, and documentation:

> **[github.com/Esturban/agent-use-cases](https://github.com/Esturban/agent-use-cases)**

Each example includes the schema, workflow, and test cases. Fork any of them as a starting point for your own implementation.

---

*Questions? Reach out at **info.evadvisory@gmail.com***